# Data Engineering Home Task Lecture 13: ETL Pipeline

Побудувати локальний ETL пайплайн, який зчитує 4 сирих CSV-файли, очищає їх від навмисних помилок, завантажує у локальну реляційну базу даних та створює аналітичні вихідні таблиці.

In [6]:
import pandas as pd
import sqlite3
import re

df_customers = pd.read_csv('customers.csv')
df_products = pd.read_csv('products.csv')
df_orders = pd.read_csv('orders.csv')
df_order_items = pd.read_csv('order_items.csv')

print("Дані успішно завантажено!")

Дані успішно завантажено!


Трансформація та очищення (Transform)

1. Таблиця Customers:

Missing IDs & Duplicates: Видаляємо записи без customer_id та дублікати (залишаємо перше входження).

Invalid/Missing Emails: Застосовуємо регулярний вираз для валідації email-адрес. Видаляємо порожні або невалідні.

Invalid timestamps: Конвертуємо дати у формат datetime. Нерелевантні дати (які pandas не може розпізнати) стають NaT і ми їх відкидаємо.

2. Таблиця Products:

Missing IDs: Видаляємо записи без product_id.

Negative/Zero Prices: Ціна не може бути нульовою чи від'ємною, оскільки це порушує бізнес-логіку. Такі товари відкидаємо.

Missing Strings: Якщо назва або категорія відсутня, заповнюємо їх значенням Unknown.

3. Таблиця Orders:

Missing IDs: Видаляємо замовлення без order_id.

Missing customer references: Відкидаємо замовлення, які посилаються на customer_id, якого більше немає у нашій очищеній таблиці клієнтів (Referential Integrity).

Mixed-case statuses & Unknown statuses: Зводимо всі статуси до нижнього регістру. Залишаємо лише валідні статуси (completed, pending, cancelled, returned).

Invalid timestamps: Аналогічно до клієнтів, парсимо та очищаємо дати.

4. Таблиця Order Items:

Missing IDs: Видаляємо порожні та дублюючі записи.

Missing order/product references: Відкидаємо записи, які посилаються на видалені замовлення або видалені продукти.

Negative quantity: Кількість товару має бути строго більшою за 0.

In [7]:
def clean_customers(df):
    df = df.dropna(subset=['customer_id']).copy()
    df = df.drop_duplicates(subset=['customer_id'], keep='first')
    
    email_regex = r'^[\w\.-]+@[\w\.-]+\.\w+$'
    valid_email_mask = df['email'].str.match(email_regex, na=False)
    df = df[valid_email_mask]
    
    df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
    df = df.dropna(subset=['created_at'])
    return df

def clean_products(df):
    df = df.dropna(subset=['product_id']).copy()
    df = df.drop_duplicates(subset=['product_id'], keep='first')
    
    df = df[df['price'] > 0]
    
    df['name'] = df['name'].fillna('Unknown')
    df['category'] = df['category'].fillna('Unknown')
    return df

def clean_orders(df, valid_customers):
    df = df.dropna(subset=['order_id']).copy()
    df = df.drop_duplicates(subset=['order_id'], keep='first')
    
    df = df.dropna(subset=['customer_id'])
    df = df[df['customer_id'].isin(valid_customers['customer_id'])]
    
    df['order_status'] = df['order_status'].astype(str).str.lower().str.strip()
    valid_statuses = ['completed', 'pending', 'cancelled', 'returned']
    df = df[df['order_status'].isin(valid_statuses)]
    
    df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
    df = df.dropna(subset=['created_at'])
    return df

def clean_order_items(df, valid_orders, valid_products):
    df = df.dropna(subset=['order_item_id']).copy()
    df = df.drop_duplicates(subset=['order_item_id'], keep='first')
    
    df = df[df['order_id'].isin(valid_orders['order_id'])]
    df = df[df['product_id'].isin(valid_products['product_id'])]
    
    df = df[df['quantity'] > 0]
    return df

cleaned_customers = clean_customers(df_customers)
cleaned_products = clean_products(df_products)
cleaned_orders = clean_orders(df_orders, cleaned_customers)
cleaned_order_items = clean_order_items(df_order_items, cleaned_orders, cleaned_products)

print(f"Клієнти: було {len(df_customers)}, стало {len(cleaned_customers)}")
print(f"Продукти: було {len(df_products)}, стало {len(cleaned_products)}")
print(f"Замовлення: було {len(df_orders)}, стало {len(cleaned_orders)}")
print(f"Деталі замовлень: було {len(df_order_items)}, стало {len(cleaned_order_items)}")

Клієнти: було 357, стало 350
Продукти: було 155, стало 152
Замовлення: було 456, стало 452
Деталі замовлень: було 914, стало 906


Завантаження в БД (Load) та створення Output Tables

In [8]:

conn = sqlite3.connect('ecommerce_dwh.db')

cleaned_customers.to_sql('dim_customers', conn, index=False, if_exists='replace')
cleaned_products.to_sql('dim_products', conn, index=False, if_exists='replace')
cleaned_orders.to_sql('fact_orders', conn, index=False, if_exists='replace')
cleaned_order_items.to_sql('fact_order_items', conn, index=False, if_exists='replace')

query_customer_spending = """
SELECT 
    c.customer_id,
    c.email,
    c.country,
    SUM(oi.quantity * p.price) as total_spent,
    COUNT(DISTINCT o.order_id) as total_completed_orders
FROM dim_customers c
JOIN fact_orders o ON c.customer_id = o.customer_id
JOIN fact_order_items oi ON o.order_id = oi.order_id
JOIN dim_products p ON oi.product_id = p.product_id
WHERE o.order_status = 'completed'
GROUP BY c.customer_id, c.email, c.country
ORDER BY total_spent DESC;
"""

query_product_sales = """
SELECT 
    p.product_id,
    p.name,
    p.category,
    SUM(oi.quantity) as total_units_sold,
    SUM(oi.quantity * p.price) as total_revenue
FROM dim_products p
JOIN fact_order_items oi ON p.product_id = oi.product_id
JOIN fact_orders o ON oi.order_id = o.order_id
WHERE o.order_status = 'completed'
GROUP BY p.product_id, p.name, p.category
ORDER BY total_revenue DESC;
"""

agg_customer_spending = pd.read_sql(query_customer_spending, conn)
agg_product_sales = pd.read_sql(query_product_sales, conn)

agg_customer_spending.to_sql('agg_customer_spending', conn, index=False, if_exists='replace')
agg_product_sales.to_sql('agg_product_sales', conn, index=False, if_exists='replace')

print("Пайплайн успішно завершено! Дані завантажено у БД ecommerce_dwh.db")

Пайплайн успішно завершено! Дані завантажено у БД ecommerce_dwh.db


Результати (Аналітичні вітрини)

In [9]:
print("--- ТОП 5 КЛІЄНТІВ ЗА ВИТРАТАМИ ---")
display(agg_customer_spending.head(5))

print("\n--- ТОП 5 ПРОДУКТІВ ЗА ДОХОДОМ ---")
display(agg_product_sales.head(5))

conn.close()

--- ТОП 5 КЛІЄНТІВ ЗА ВИТРАТАМИ ---


,customer_id,email,country,total_spent,total_completed_orders
0,335.0,user335@example.com,UA,22803.45,5
1,204.0,user204@example.com,ES,21745.49,4
2,111.0,user111@example.com,US,20527.57,2
3,323.0,user323@example.com,ES,20349.01,3
4,89.0,user89@example.com,CZ,20332.38,4



--- ТОП 5 ПРОДУКТІВ ЗА ДОХОДОМ ---


,product_id,name,category,total_units_sold,total_revenue
0,1122,Toys Product 1122,Toys,31,38068.00
1,1039,Books Product 1039,Books,22,30991.40
2,1047,Electronics Product 1047,Electronics,20,28472.40
3,1008,Books Product 1008,Books,17,24554.97
4,1142,Books Product 1142,Books,22,21858.76
